<a href="https://colab.research.google.com/github/getcommunityone/open-navigator/blob/main/scripts/colab/02_run_meeting_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 4 Good — Municipal Meeting Intelligence

**One-liner:** Real Tuscaloosa & Big Timber government records → Gemma 4 reads scanned PDFs, spends vision tokens wisely, reasons over policy, and tracks how an issue drifts across a long meeting.

## For judges (≈45 min)

1. Add Colab secret **`GEMINI_API_KEY`** ([AI Studio](https://aistudio.google.com/apikey)).
2. Optional: **`HF_TOKEN`** only if you set `GOVERNANCE_GATEKEEPER_FORCE_HF=1` (local E2B weights).
3. **Runtime → Run all** (DEMO mode: last **3 meeting dates**, 3 PDFs/jurisdiction, 4 audio chunks).
4. Results on Drive: `My Drive/CommunityOne/hackathons/2026_Gemma_4_Good/03_processed_outputs/`

| Step | What it proves | Look here |
|------|----------------|-----------|
| **§5 Pipeline** | Gatekeeper + OCR + token budget + policy JSON + audio drift | `03_processed_outputs/` (per jurisdiction) |
| Gatekeeper | Junk filtered before heavy LLM work | `_gatekeeper/triage_report_*.json` |
| Visual OCR | Scanned minutes with no text layer | `02_gemma_json/.../*visual_ocr*` |
| Token budget | HIGH vs LOW tokens per page | `02_gemma_json/.../_token_budget_report.json` |
| Reasoning | Policy deconstruction trace | `*.thinking.thoughts.md` |
| Audio drift | Issue movement across chunks | `policy_drift.json` + `policy_drift.mmd` |

<details>
<summary>Optional: model & backend details</summary>

- **Default (fast):** Gatekeeper + demos on **Google AI Studio** (`gemma-3n-e2b-it` triage, `gemma-4-26b-a4b-it` demos).
- **Optional HF E2B:** `GOVERNANCE_GATEKEEPER_FORCE_HF=1` + `HF_TOKEN` loads [google/gemma-4-E2B](https://huggingface.co/google/gemma-4-E2B) locally (~10GB, slow start).
- [Gemma 4 model card](https://ai.google.dev/gemma/docs/core/model_card_4).

</details>


## Appendix: models (optional)

| Role | Default backend | Default id | Env var |
|------|-----------------|------------|--------|
| Gatekeeper triage | **Google AI Studio** | `gemma-3n-e2b-it` | `GOVERNANCE_GATEKEEPER_MODEL` |
| Gatekeeper (HF E2B) | Hugging Face (~10GB) | `google/gemma-4-E2B-it` | `GOVERNANCE_GATEKEEPER_FORCE_HF=1` |
| Demos 1–5 | Google AI Studio | `gemma-4-26b-a4b-it` | `GOVERNANCE_GENAI_MODEL` |

Section 3 prints resolved ids.


## Before you run

1. **Secret:** Colab 🔑 → `GEMINI_API_KEY` ([AI Studio](https://aistudio.google.com/apikey)).
2. **HF_TOKEN** only if you opt into local E2B (`GOVERNANCE_GATEKEEPER_FORCE_HF=1`).
3. **Drive:** `My Drive/CommunityOne/hackathons/2026_Gemma_4_Good/`.
4. **Run order:** §1 → §2 → §3 → Gatekeeper §4 → demos. **No restart** needed unless §2 upgraded transformers for FORCE_HF.


## 1) Bootstrap (Drive + repo)

Mounts Drive, pulls latest `open-navigator`, resolves the pipeline folder on Drive. Prints **Repo** and **Governance data root** — confirm the hackathon path, not `/content/...`.


In [ ]:
# Find the open-navigator repo, refresh it from GitHub, then import the resolver.
# Order matters: any cached scripts.colab.* modules must be evicted BEFORE the
# import below, or git-reset --hard won't take effect this session.
import os
import shutil
import sys
from pathlib import Path

REPO_PATH = Path("/content/open-navigator")
EPHEMERAL_DATA_DIR = Path("/content/governance_pipeline_data")

# 1) Clone if missing.
if not REPO_PATH.is_dir():
    print(f"Cloning open-navigator into {REPO_PATH}...")
    rc = os.system(f"git clone https://github.com/getcommunityone/open-navigator.git {REPO_PATH}")
    if rc != 0:
        raise RuntimeError(f"git clone failed (exit {rc})")

os.environ.setdefault("OPEN_NAVIGATOR_ROOT", str(REPO_PATH))

# 2) git fetch + reset --hard origin/main BEFORE the first import, so Python loads
#    the latest colab_paths.py from disk (not a cached stale version).
RUN_GIT_UPDATE = True
if RUN_GIT_UPDATE and (REPO_PATH / ".git").is_dir():
    print("🔄 Fetching latest open-navigator from GitHub...")
    rc = os.system(
        f"cd {REPO_PATH} && git config core.hooksPath /dev/null "
        "&& git fetch origin && git reset --hard origin/main"
    )
    if rc != 0:
        print(f"⚠️  git update returned exit {rc} — continuing with whatever is on disk.")

# 3) Defensive cleanup of stale session state from prior runs.
#    Without this, re-running the cell silently uses cached old code / paths.
for _mod in [m for m in list(sys.modules) if m.startswith("scripts.colab") or m.startswith("scripts.utils.gdrive_paths")]:
    sys.modules.pop(_mod, None)
_stale_root = os.environ.pop("GOVERNANCE_PIPELINE_DATA_ROOT", None)
if _stale_root:
    print(f"   Cleared stale GOVERNANCE_PIPELINE_DATA_ROOT={_stale_root}")

# 4) Remove the ephemeral /content/governance_pipeline_data shell if a previous
#    notebook version (or PIPE.ensure_dirs) left it behind. Real data lives on
#    mounted Drive — this folder is on disposable runtime disk and confuses judges.
if EPHEMERAL_DATA_DIR.is_dir():
    print(f"   Removing stale ephemeral folder {EPHEMERAL_DATA_DIR} "
          "(real data lives on Drive — this is just an empty shell from a prior bug).")
    shutil.rmtree(EPHEMERAL_DATA_DIR, ignore_errors=True)

# 5) Put the repo on sys.path so `from scripts.colab.colab_paths import …` works.
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# 6) Now the first import sees the freshly-pulled code.
from scripts.colab.colab_paths import maybe_mount_google_drive, setup_notebook_paths

maybe_mount_google_drive()

try:
    PATHS = setup_notebook_paths()
except RuntimeError as e:
    # Resolver couldn't find the data root on Drive. Surface its diagnostic +
    # show what IS visible under the mount so the user can spot the typo.
    print("\n❌ setup_notebook_paths() failed:\n")
    print(str(e))
    print("\nCurrent /content/drive view:")
    for p in (Path("/content/drive"), Path("/content/drive/MyDrive"),
              Path("/content/drive/MyDrive/CommunityOne"),
              Path("/content/drive/MyDrive/CommunityOne/hackathons")):
        if p.is_dir():
            children = sorted([c.name for c in p.iterdir()])[:20]
            print(f"  {p}: {children}")
        else:
            print(f"  {p}: (missing)")
    raise

print("Runtime:", "Google Colab" if PATHS.in_colab else "Local")
print("Repo:                 ", PATHS.project_path)
print("Governance data root: ", PATHS.governance_pipeline_data)


In [ ]:
import os
import sys

# Repo path is fixed in cell 4 (git fetch + reset --hard ran there).
proj = PATHS.project_path

assert (proj / "prompts" / "policy_analysis_v1.md").is_file(), (
    f"Missing {proj}/prompts/policy_analysis_v1.md"
)

sys.path.insert(0, str(proj))
COLAB_DIR = proj / "scripts" / "colab"
sys.path.insert(0, str(COLAB_DIR))

# Drop stale Colab copies of scripts/colab/*.py so imports pick up a fresh git pull.
for _mod in list(sys.modules):
    if _mod in ("gatekeeper_triage", "governance_meeting_llm", "gemma_hf_backend"):
        del sys.modules[_mod]

# Pin the resolved Drive path so downstream modules read it from the env.
os.environ.setdefault("GOVERNANCE_PIPELINE_DATA_ROOT", str(PATHS.governance_pipeline_data))

from scripts.utils.gdrive_paths import GovernancePipelinePaths

PIPE = GovernancePipelinePaths.resolve()
# Create only the subfolders we will write into (03_processed_outputs/*). We do
# NOT create 01_raw_inputs here — if the resolver pointed somewhere wrong, an
# empty 01_raw_inputs would mask the bug.
for _sub in (PIPE.transcripts, PIPE.gemma_json, PIPE.human_summaries):
    _sub.mkdir(parents=True, exist_ok=True)

from governance_meeting_llm import (
    AUDIO_EXTS,
    PDF_EXTS,
    TOKEN_BUDGET_HIGH,
    TOKEN_BUDGET_LOW,
    call_google_genai_multimodal,
    transcribe_audio_with_gemma,
    TRANSCRIPTION_SUPPORTED_LANGUAGES,
    chunk_audio_ffmpeg,
    classify_pdf_page_heuristic,
    extract_pdf_digital_text,
    is_scanned_pdf,
    load_contacts_lookup,
    load_meeting_data_lookup,
    load_text_file,
    merge_contacts_by_jurisdiction,
    merge_meeting_data_by_jurisdiction,
    mime_for,
    mirror_output_path,
    parse_policy_analysis_response,
    policy_drift_summarize,
    render_pdf_pages,
    walk_raw_inputs,
    embed_text_with_gemma,
    cosine_similarity_matrix,
    shield_review_text,
)

PROMPT_PATH = proj / "prompts" / "policy_analysis_v1.md"
POLICY_PROMPT = load_text_file(PROMPT_PATH)
print("Loaded prompt chars:", len(POLICY_PROMPT))
print("Pipeline data root:", PIPE.root)


## 2) Install dependencies

Lightweight install only. **Gatekeeper defaults to Google AI Studio** (no ~10GB model download).
§3 installs `transformers` only when `GOVERNANCE_GATEKEEPER_FORCE_HF=1`.


In [ ]:
# poppler-utils for pdf2image (Gatekeeper). Skipped off Colab.
import shutil, subprocess, sys
if not shutil.which("pdftoppm"):
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "poppler-utils"], check=False)
    except FileNotFoundError:
        pass

subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "google-genai>=1.0", "librosa", "pymupdf", "pdf2image",
    ]
)
print("Installed google-genai, pymupdf, pdf2image (Gatekeeper uses AI Studio by default — no transformers download).")
print("If GOVERNANCE_GATEKEEPER_FORCE_HF=1, §3 will install transformers>=5.5.0 and load E2B.")


## 3) API keys, models & DEMO caps

One cell: backend, secrets, model ids, and **DEMO limits** (Gatekeeper **last 3 meeting dates**).  
Set `GOVERNANCE_LOG_LLM_CATALOG=1` only if you need the full `models.list()` dump.


In [ ]:
import os

# Judge-friendly defaults (edit here only if needed)
os.environ.setdefault("GOVERNANCE_MODE", "DEMO")
os.environ.setdefault("GOVERNANCE_LLM_BACKEND", "google")
os.environ.setdefault("GOVERNANCE_LOG_LLM_CATALOG", "0")
os.environ.setdefault("GOVERNANCE_ORGANIZE_MEETINGS", "1")
os.environ.setdefault("GOVERNANCE_PARALLEL_STATES", "2")
os.environ.setdefault("GOVERNANCE_DEMO_DATE_SCOPE", "1")
os.environ.setdefault("GOVERNANCE_DEMO_MEETING_DATES", "3")
os.environ.setdefault("GOVERNANCE_DEMO_YEAR_SCOPE", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PDF_PAGES", "2")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PDF_DPI", "120")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_COPY_TO_TMP", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_LARGE_PDF_BYTES", "1500000")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_TEXT_FIRST", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_TEXT_MIN_CHARS", "80")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_FSYNC", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_API_TIMEOUT_SECONDS", "120")
# Fast Gatekeeper triage (demos use GOVERNANCE_GENAI_MODEL / 26B).
os.environ["GOVERNANCE_GATEKEEPER_MODEL"] = "gemma-3n-e2b-it"
# os.environ.setdefault("GOVERNANCE_GATEKEEPER_FORCE_HF", "1")  # opt-in: ~10GB HF download
# os.environ["GOVERNANCE_GATEKEEPER_MODEL"] = "gemma-4-26b-a4b-it"  # slow triage; only if small ids missing

LLM_BACKEND = os.environ.get("GOVERNANCE_LLM_BACKEND", "google").strip().lower()
USE_HF_ALL = LLM_BACKEND in ("huggingface", "hf", "local")

try:
    from google.colab import userdata
    _get_secret = userdata.get
except ImportError:
    def _get_secret(name: str):
        return os.environ.get(name)


def _sanitize_key(value):
    if not value:
        return value
    cleaned = (
        value.strip().replace("\r", "").replace("\n", "").replace("\t", "").replace(" ", "")
    )
    if cleaned != value.strip():
        print(f"   ⚠️  API key whitespace stripped — re-copy Colab Secret to silence this.")
    return cleaned


HF_TOKEN = _sanitize_key(_get_secret("HF_TOKEN"))
GEMINI_API_KEY = _sanitize_key(_get_secret("GEMINI_API_KEY") or _get_secret("GOOGLE_API_KEY"))

if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY (Colab Secret). https://aistudio.google.com/apikey")
if not GEMINI_API_KEY.startswith("AIza") or len(GEMINI_API_KEY) < 35:
    raise RuntimeError(f"GEMINI_API_KEY looks malformed (len={len(GEMINI_API_KEY)}).")
API_KEY = GEMINI_API_KEY

GENAI_MODEL = os.environ.get("GOVERNANCE_GENAI_MODEL", "gemma-4-26b-a4b-it").strip()
GATEKEEPER_MODEL = os.environ.get("GOVERNANCE_GATEKEEPER_MODEL", "gemma-3n-e2b-it").strip()
EMBEDDING_MODEL = os.environ.get("GOVERNANCE_EMBEDDING_MODEL", "embeddinggemma-300m").strip()
SHIELD_MODEL = os.environ.get("GOVERNANCE_SHIELD_MODEL", "shieldgemma-9b").strip() or GENAI_MODEL

from gemma_hf_backend import (
    gatekeeper_use_huggingface,
    model_requires_huggingface,
    preload_gemma_hf,
    print_hf_model_catalog,
    resolve_hf_model_id,
    ensure_transformers_gemma4_ready,
    _HF_GATEKEEPER_REPO_DEFAULT,
)

NEEDS_HF = (
    USE_HF_ALL
    or model_requires_huggingface(GENAI_MODEL)
    or gatekeeper_use_huggingface()
)
if NEEDS_HF:
    if not HF_TOKEN:
        raise RuntimeError(
            "HF_TOKEN required when GOVERNANCE_GATEKEEPER_FORCE_HF=1 or GOVERNANCE_LLM_BACKEND=huggingface. "
            "Default Gatekeeper uses AI Studio only — no HF_TOKEN needed."
        )
    os.environ["HF_TOKEN"] = HF_TOKEN

_log_llm_catalog = os.environ.get("GOVERNANCE_LOG_LLM_CATALOG", "0") == "1"

from gatekeeper_triage import (
    _GEMMA_HEAVY_FALLBACKS,
    _GEMMA_GATEKEEPER_AI_FALLBACKS,
    _GEMMA_TRIAGE_FALLBACKS,
    _build_genai_client,
    print_available_models,
    resolve_model_id,
)

_resolver_client = _build_genai_client(GEMINI_API_KEY)
if _log_llm_catalog:
    print_available_models(
        _resolver_client,
        requested=(GENAI_MODEL, GATEKEEPER_MODEL, EMBEDDING_MODEL, SHIELD_MODEL),
        role="Google AI Studio",
    )

if model_requires_huggingface(GENAI_MODEL):
    GENAI_MODEL = resolve_hf_model_id(GENAI_MODEL)
    ensure_transformers_gemma4_ready(auto_install=True)
    if _log_llm_catalog:
        print_hf_model_catalog(requested=(GENAI_MODEL,), role="Hugging Face (demos)")
    print(f"Loading HF weights for demos: {GENAI_MODEL} …")
    preload_gemma_hf(GENAI_MODEL, load_audio_variant=True)
else:
    GENAI_MODEL = resolve_model_id(
        _resolver_client, GENAI_MODEL, fallbacks=_GEMMA_HEAVY_FALLBACKS, role="Demos (AI Studio)"
    )

if gatekeeper_use_huggingface():
    ensure_transformers_gemma4_ready(auto_install=True)
    hf_gate = (
        os.environ.get("GOVERNANCE_GATEKEEPER_MODEL", "").strip()
        or os.environ.get("GOVERNANCE_HF_GATEKEEPER_MODEL", "").strip()
        or _HF_GATEKEEPER_REPO_DEFAULT
    )
    GATEKEEPER_MODEL = resolve_hf_model_id(hf_gate)
    if _log_llm_catalog:
        print_hf_model_catalog(requested=(GATEKEEPER_MODEL,), role="Hugging Face (Gatekeeper)")
    print(f"Loading HF weights for Gatekeeper: {GATEKEEPER_MODEL} … (~10GB first time)")
    preload_gemma_hf(GATEKEEPER_MODEL, load_audio_variant=False)
else:
    GATEKEEPER_MODEL = resolve_model_id(
        _resolver_client,
        GATEKEEPER_MODEL,
        fallbacks=_GEMMA_GATEKEEPER_AI_FALLBACKS,
        role="Gatekeeper (AI Studio)",
    )

EMBEDDING_MODEL = resolve_model_id(
    _resolver_client,
    EMBEDDING_MODEL,
    fallbacks=("embeddinggemma-300m", "text-embedding-004", "gemini-embedding-001"),
    role="Embedding model",
)
SHIELD_MODEL = resolve_model_id(
    _resolver_client,
    SHIELD_MODEL,
    fallbacks=("shieldgemma-9b", "shieldgemma-2b", *_GEMMA_TRIAGE_FALLBACKS),
    role="Shield model",
)

MAX_PDFS_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_PDFS_PER_JUR", "3"))
MAX_PAGES_PER_PDF = int(os.environ.get("GOVERNANCE_DEMO_MAX_PAGES_PER_PDF", "8"))
MAX_AUDIO_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_PER_JUR", "1"))
MAX_AUDIO_CHUNKS = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_CHUNKS", "4"))
THINKING_BUDGET = int(os.environ.get("GOVERNANCE_DEMO_THINKING_BUDGET", "-1"))
DRIFT_FOCUS = os.environ.get("GOVERNANCE_DRIFT_FOCUS", "").strip() or None
_gate_max_env = os.environ.get("GOVERNANCE_GATEKEEPER_MAX_FILES", "").strip()
GATEKEEPER_MAX_FILES = int(_gate_max_env) if _gate_max_env else None


def _backend_label(model: str, *, gatekeeper: bool = False) -> str:
    from gemma_hf_backend import gatekeeper_use_huggingface, model_requires_huggingface
    if gatekeeper:
        return "Hugging Face" if gatekeeper_use_huggingface() else "Google AI Studio"
    return "Hugging Face" if model_requires_huggingface(model) else "Google AI Studio"


print("LLM mode:", "all Hugging Face" if USE_HF_ALL else "hybrid (AI Studio default)")
print("GenAI model (demos):     ", GENAI_MODEL, f"({_backend_label(GENAI_MODEL)})")
print(
    "Gatekeeper (triage):     ",
    GATEKEEPER_MODEL,
    f"({_backend_label(GATEKEEPER_MODEL, gatekeeper=True)})",
)
if gatekeeper_use_huggingface():
    print("  (GOVERNANCE_GATEKEEPER_FORCE_HF=1 — local E2B weights)")
print("Embedding (demo 6):      ", EMBEDDING_MODEL, "(Google AI Studio)")
print("Shield (safety pass):    ", SHIELD_MODEL, "(Google AI Studio)")
print(
    "Caps:",
    f"pdfs/jur={MAX_PDFS_PER_JUR}",
    f"pages/pdf={MAX_PAGES_PER_PDF}",
    f"audio/jur={MAX_AUDIO_PER_JUR}",
    f"chunks/audio={MAX_AUDIO_CHUNKS}",
    f"thinking_budget={THINKING_BUDGET}",
    f"gatekeeper_max_files={GATEKEEPER_MAX_FILES if GATEKEEPER_MAX_FILES is not None else 'unlimited'}",
)


## 4) Inventory

Lists pilot jurisdictions under `01_raw_inputs/`. §5 runs Gatekeeper and demos 1–4 for each place before moving on.


In [ ]:
from pathlib import Path
import os

RAW_ROOT = Path(
    os.environ.get("GOVERNANCE_RAW_INPUTS_ROOT", "").strip() or (PIPE.root / "01_raw_inputs")
).expanduser()
PROCESSED_ROOT = PIPE.root / "03_processed_outputs"
GEMMA_JSON_ROOT = PROCESSED_ROOT / "02_gemma_json"
SUMMARIES_ROOT = PROCESSED_ROOT / "03_human_summaries"
SCRATCH_AUDIO_ROOT = PROCESSED_ROOT / "_scratch_audio_chunks"

for p in (GEMMA_JSON_ROOT, SUMMARIES_ROOT, SCRATCH_AUDIO_ROOT):
    p.mkdir(parents=True, exist_ok=True)

if not RAW_ROOT.is_dir():
    raise FileNotFoundError(
        f"Raw inputs root missing: {RAW_ROOT}. Run 01_copy_scraped_meetings_cache_to_gdrive.py "
        "or set GOVERNANCE_RAW_INPUTS_ROOT."
    )

try:
    from meeting_date_scope import resolve_demo_meeting_dates_limit
    _demo_date_cap = resolve_demo_meeting_dates_limit()
except ImportError:
    _demo_date_cap = None

INVENTORIES = [inv for inv in walk_raw_inputs(RAW_ROOT) if inv.has_media]

print(f"Raw inputs root: {RAW_ROOT}")
print(f"Jurisdictions with media: {len(INVENTORIES)}")
_states = sorted({inv.jurisdiction.state_code for inv in INVENTORIES})
if _states:
    print(f"States: {', '.join(_states)}")
for inv in INVENTORIES:
    j = inv.jurisdiction
    print(
        f"  {j.relative_label}  fips={j.fips or '—'}  "
        f"pdfs={len(inv.pdfs)}  audio={len(inv.audio)}  images={len(inv.images)}"
    )

if not INVENTORIES:
    print(
        "\nNo media found. Drop PDFs / audio under "
        f"{RAW_ROOT}/<STATE>/<county|municipality>/<jurisdiction_slug>/…"
    )


## 5) Run pipeline (per jurisdiction)

Each jurisdiction: **Gatekeeper** → meeting folders → **demos 1–4** (OCR, token budget, policy JSON, audio drift). When you have multiple states, up to **`GOVERNANCE_PARALLEL_STATES`** (default `2`) run at once; places within a state run one after another.

Logs: `03_processed_outputs/_gatekeeper/triage_<jurisdiction>_*.txt`


In [ ]:
import importlib
import os

import colab_demos
import jurisdiction_pipeline as _jp
importlib.reload(colab_demos)
importlib.reload(_jp)

from colab_demos import DemoContext
from governance_meeting_llm import inventory_for_jurisdiction
from jurisdiction_pipeline import JurisdictionRunContext, run_governance_pipeline

if not INVENTORIES:
    print("No inventories — skip pipeline.")
else:
    _demo_ctx = DemoContext(
        api_key=API_KEY,
        genai_model=GENAI_MODEL,
        raw_root=RAW_ROOT,
        processed_root=PROCESSED_ROOT,
        gemma_json_root=GEMMA_JSON_ROOT,
        summaries_root=SUMMARIES_ROOT,
        scratch_audio_root=SCRATCH_AUDIO_ROOT,
        policy_prompt=POLICY_PROMPT,
        max_pdfs_per_jur=MAX_PDFS_PER_JUR,
        max_pages_per_pdf=MAX_PAGES_PER_PDF,
        max_audio_per_jur=MAX_AUDIO_PER_JUR,
        max_audio_chunks=MAX_AUDIO_CHUNKS,
        thinking_budget=THINKING_BUDGET,
        drift_focus=DRIFT_FOCUS,
    )
    _run_ctx = JurisdictionRunContext(
        raw_root=RAW_ROOT,
        pipe_root=PIPE.root,
        api_key=API_KEY,
        gatekeeper_model=GATEKEEPER_MODEL,
        demo_ctx=_demo_ctx,
        demo_date_cap=_demo_date_cap,
        gatekeeper_max_files=GATEKEEPER_MAX_FILES,
    )
    run_governance_pipeline(INVENTORIES, _run_ctx)
    INVENTORIES = [
        inventory_for_jurisdiction(RAW_ROOT, inv.jurisdiction.root) or inv
        for inv in INVENTORIES
    ]
    if _demo_date_cap:
        from meeting_date_scope import filter_inventory_media
        for inv in INVENTORIES:
            inv.pdfs, inv.audio = filter_inventory_media(
                inv.pdfs, inv.audio, RAW_ROOT, inv.jurisdiction.root, max_dates=_demo_date_cap
            )
    print(f"\nPipeline complete. Optional cells below use refreshed inventories.")


## 6) Plain transcript *(optional)*

Word-for-word transcript per chunk beside Demo 4 JSON. Default language: `en` (`GOVERNANCE_TRANSCRIPTION_LANGUAGES`).


In [ ]:
# Demo 4a — Plain Transcription per audio file.
# Mirrors Demo 4's audio iteration so transcripts land beside the chunk JSONs.
_TRANSCRIPTION_LANGS_RAW = os.environ.get("GOVERNANCE_TRANSCRIPTION_LANGUAGES", "en")
TRANSCRIPTION_LANGS = [
    lang.strip()
    for lang in _TRANSCRIPTION_LANGS_RAW.split(",")
    if lang.strip()
]
# Validate up-front so a typo in the env var fails fast, not after N audio calls.
_SUPPORTED_LANG_TAGS = {tag.lower() for tag in TRANSCRIPTION_SUPPORTED_LANGUAGES.values()}
_invalid_langs = [
    lang for lang in TRANSCRIPTION_LANGS if lang.lower() not in _SUPPORTED_LANG_TAGS
]
if _invalid_langs:
    raise RuntimeError(
        f"GOVERNANCE_TRANSCRIPTION_LANGUAGES contains unsupported tag(s) {_invalid_langs}. "
        f"Supported: {sorted(_SUPPORTED_LANG_TAGS)} (currently {list(TRANSCRIPTION_SUPPORTED_LANGUAGES)})."
    )

_TRANSCRIPTION_MAX_CHUNKS = int(
    os.environ.get("GOVERNANCE_TRANSCRIPTION_MAX_CHUNKS", str(MAX_AUDIO_CHUNKS))
)

demo4a_report = []
if not TRANSCRIPTION_LANGS:
    print("Demo 4a: GOVERNANCE_TRANSCRIPTION_LANGUAGES is empty — skipping.")
else:
    print(f"Demo 4a: transcribing in {TRANSCRIPTION_LANGS} (cap {_TRANSCRIPTION_MAX_CHUNKS} chunks/file)")
    for inv in INVENTORIES:
        j = inv.jurisdiction
        audios = inv.audio[:MAX_AUDIO_PER_JUR]
        if not audios:
            continue
        print(f"\n— {j.relative_label}: {len(audios)} audio file(s)")
        for audio in audios:
            print(f"  • {audio.name}")
            per_audio_dir = mirror_output_path(
                input_path=audio, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT, suffix="",
            )
            per_audio_dir.mkdir(parents=True, exist_ok=True)

            rel = audio.resolve().relative_to(RAW_ROOT.resolve())
            scratch = SCRATCH_AUDIO_ROOT / rel.with_suffix("")
            scratch.mkdir(parents=True, exist_ok=True)

            # Reuse Demo 4 chunks if they're already on disk; otherwise chunk now.
            existing = sorted(scratch.glob("chunk_*.mp3"))
            if existing:
                chunks = existing
                print(f"    reusing {len(chunks)} existing chunk(s) from {scratch.name}/")
            else:
                try:
                    chunks = chunk_audio_ffmpeg(audio, out_dir=scratch, chunk_minutes=15, fmt="mp3")
                except Exception as e:
                    print(f"    ! ffmpeg chunking failed: {e}")
                    continue
            chunks = chunks[:_TRANSCRIPTION_MAX_CHUNKS]
            print(f"    {len(chunks)} chunk(s) to transcribe")

            for lang in TRANSCRIPTION_LANGS:
                pieces = []
                errors = 0
                for idx, chunk_path in enumerate(chunks):
                    try:
                        text = transcribe_audio_with_gemma(
                            api_key=API_KEY,
                            model=GENAI_MODEL,
                            audio_path=chunk_path,
                            language=lang,
                            mime_type=mime_for(chunk_path),
                        )
                    except Exception as e:
                        errors += 1
                        print(f"    ! chunk {idx} [{lang}] failed: {e}")
                        continue
                    if text:
                        pieces.append(text)

                if not pieces:
                    print(f"    [{lang}] no text produced — skipping write")
                    continue

                # Join chunk transcripts with single spaces. The model is instructed
                # not to emit newlines; we still collapse defensively in the helper.
                transcript = " ".join(pieces)
                out_path = per_audio_dir / f"transcript.{lang}.txt"
                out_path.write_text(transcript, encoding="utf-8")
                print(
                    f"    [{lang}] → {out_path.relative_to(PROCESSED_ROOT)} "
                    f"({len(transcript):,} chars from {len(pieces)}/{len(chunks)} chunks"
                    + (f", {errors} error(s)" if errors else "")
                    + ")"
                )
                demo4a_report.append(
                    {
                        "jurisdiction": j.relative_label,
                        "audio": str(audio.relative_to(RAW_ROOT)),
                        "language": lang,
                        "chunks_transcribed": len(pieces),
                        "chunks_attempted": len(chunks),
                        "errors": errors,
                        "chars": len(transcript),
                    }
                )

if demo4a_report:
    print(
        f"\nDemo 4a done: {len(demo4a_report)} transcript file(s) written. "
        "Plain text is now beside each audio's chunk JSONs for citation / human review."
    )
else:
    print("\nDemo 4a: no transcripts written (no audio in inventory or all chunks failed).")

## 7) Contact photos *(optional)*

Classifies `_contact_images/` as person vs. logo/building; writes JSON beside each image.


In [ ]:
# Demo 5 — Contact image enrichment. For each image found under a jurisdiction,
# decide whether it is a person; return perceived demographics or a subject tag.
import json as _json

DEMO5_SYSTEM = (
    "You are a careful image triage assistant for a local-government open-data "
    "pipeline. You analyze public contact photos of elected officials and "
    "department heads scraped from public government websites. You always return "
    "strict JSON only. You never claim certainty about demographic attributes — "
    "every attribute is explicitly perceived from the image and you return null "
    "when the image is too low-resolution, partially occluded, ambiguous, or when "
    "you would be guessing rather than observing."
)
DEMO5_USER = (
    "Look at the attached image and decide whether it depicts a single identifiable "
    "person. Return JSON with this shape:\n\n"
    "{\n"
    "  \"is_person\": bool,\n"
    "  \"confidence\": float between 0.0 and 1.0,\n"
    "  \"perceived\": {\n"
    "    \"age_range\": \"one of: child, teen, 20-29, 30-39, 40-49, 50-59, 60-69, 70-plus, or null\",\n"
    "    \"race\": \"perceived racial category or null\",\n"
    "    \"gender\": \"perceived gender presentation or null\",\n"
    "    \"ethnicity\": \"perceived ethnic background or null\",\n"
    "    \"demeanor\": \"one of: formal, smiling, neutral, stern, candid, or null\"\n"
    "  } or null when is_person is false,\n"
    "  \"subject_tag\": \"short descriptor when is_person is false (e.g. 'city logo', "
    "'aerial photo of courthouse', 'organizational chart'); otherwise null\",\n"
    "  \"notes\": \"short caveat — e.g. 'low resolution', 'side profile', "
    "'multiple people visible', 'official portrait crop'\"\n"
    "}\n\n"
    "Rules:\n"
    "- These are PUBLIC officials in a PUBLIC-records transparency context.\n"
    "- Every 'perceived.*' field is a best visual estimate, not a factual claim.\n"
    "- Return null for any 'perceived' subfield where you would be guessing.\n"
    "- If multiple people appear, set is_person=false and use subject_tag='group photo' (and notes).\n"
    "- Return only the JSON object, no prose or markdown."
)


def _mime_for_image(p):
    ext = p.suffix.lower()
    if ext in (".jpg", ".jpeg"): return "image/jpeg"
    if ext == ".png": return "image/png"
    if ext == ".webp": return "image/webp"
    if ext == ".gif": return "image/gif"
    if ext == ".bmp": return "image/bmp"
    return "application/octet-stream"


MAX_IMAGES_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_IMAGES_PER_JUR", "12"))

demo5_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    if not inv.images:
        continue
    images = inv.images[:MAX_IMAGES_PER_JUR]
    print(f"\n— {j.relative_label}: {len(images)} image(s) (cap={MAX_IMAGES_PER_JUR})")
    for img in images:
        try:
            result = call_google_genai_multimodal(
                api_key=API_KEY,
                model=GENAI_MODEL,
                system_instruction=DEMO5_SYSTEM,
                user_text=DEMO5_USER,
                media=[(img, _mime_for_image(img))],
                temperature=0.0,
                max_output_tokens=1024,
                media_resolution=TOKEN_BUDGET_HIGH,
            )
        except Exception as e:
            print(f"  ! {img.name}: Gemma call failed — {e}")
            continue

        try:
            payload = _json.loads((result.text or "").strip().lstrip("`"))
        except Exception:
            payload = {"_parse_error": True, "_raw": (result.text or "")[:2000]}

        out_path = mirror_output_path(
            input_path=img, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".image_triage.json",
        )
        out_path.write_text(
            _json.dumps(
                {
                    "image": str(img.relative_to(RAW_ROOT)),
                    "jurisdiction_id": j.jurisdiction_id,
                    "model": GENAI_MODEL,
                    "result": payload,
                },
                indent=2, ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        is_person = bool(payload.get("is_person")) if isinstance(payload, dict) else False
        if is_person:
            perceived = payload.get("perceived") or {}
            details = ", ".join(
                f"{k}={v}" for k, v in perceived.items() if v not in (None, "")
            ) or "(no observable attributes)"
            print(f"  · {img.name}: PERSON — {details}")
        else:
            tag = payload.get("subject_tag") if isinstance(payload, dict) else None
            print(f"  · {img.name}: non-person → {tag or '(no tag)'}")
        demo5_report.append(
            {
                "jurisdiction": j.relative_label,
                "jurisdiction_id": j.jurisdiction_id,
                "image": str(img.relative_to(RAW_ROOT)),
                "is_person": is_person,
                "output": str(out_path.relative_to(PROCESSED_ROOT)),
            }
        )

if demo5_report:
    persons = sum(1 for r in demo5_report if r["is_person"])
    print(
        f"\nDemo 5 done: {len(demo5_report)} images classified across "
        f"{len({r['jurisdiction'] for r in demo5_report})} jurisdictions. "
        f"{persons} persons / {len(demo5_report) - persons} non-person subjects."
    )
else:
    print("\nDemo 5: no images in the inventory (check _contact_images/ folders).")

## 8) Merge reference JSON *(optional)*

Joins Gemma outputs with meeting-data / contacts reference files by `jurisdiction_id`.


In [ ]:
# Walk every *.json under GEMMA_JSON_ROOT and attach the matching meeting-data
# and contacts rows by jurisdiction_id, derived from the mirrored output path.
import json as _json


def _jurisdiction_id_from_output(path):
    """Mirror layout is <STATE>/<scope>/<jur_slug>/...  →  jurisdiction_<state>_<scope>_<tail>."""
    try:
        rel = path.resolve().relative_to(GEMMA_JSON_ROOT.resolve())
    except ValueError:
        return None
    parts = rel.parts
    if len(parts) < 3:
        return None
    state, scope, slug = parts[0].lower(), parts[1].lower(), parts[2].lower()
    scope_prefix = f"{scope}_"
    tail = slug[len(scope_prefix):] if slug.startswith(scope_prefix) else slug
    return f"jurisdiction_{state}_{scope}_{tail}"


meeting_data = load_meeting_data_lookup(PIPE.meeting_data_by_jurisdiction_id)
contacts = load_contacts_lookup(PIPE.contacts_by_jurisdiction_id)

if not meeting_data and not contacts:
    print(
        f"No reference lookups found under {PIPE.meeting_data_by_jurisdiction_id} "
        f"or {PIPE.contacts_by_jurisdiction_id} — skip."
    )
else:
    print(
        f"Loaded {len(meeting_data)} meeting_data and {len(contacts)} contacts "
        "rows keyed by jurisdiction_id."
    )
    merged_meeting = 0
    merged_contacts = 0
    missing_both = 0
    skipped = 0
    for jp in sorted(GEMMA_JSON_ROOT.rglob("*.json")):
        if jp.name.startswith("_") or jp.name.startswith("policy_drift"):
            continue
        try:
            data = _json.loads(jp.read_text(encoding="utf-8"))
        except _json.JSONDecodeError:
            skipped += 1
            continue
        jid = _jurisdiction_id_from_output(jp)
        if not jid:
            continue
        analysis = data
        if isinstance(data, dict) and "json_analysis" in data and isinstance(data["json_analysis"], dict):
            analysis = data["json_analysis"]
        if not isinstance(analysis, dict):
            continue
        had_meeting = jid in meeting_data
        had_contacts = jid in contacts
        if not (had_meeting or had_contacts):
            missing_both += 1
            continue
        if had_meeting:
            merge_meeting_data_by_jurisdiction(analysis, meeting_data, jid)
            merged_meeting += 1
        if had_contacts:
            merge_contacts_by_jurisdiction(analysis, contacts, jid)
            merged_contacts += 1
        jp.write_text(_json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        tags = []
        if had_meeting:
            tags.append("meeting_data")
        if had_contacts:
            tags.append("contacts")
        print(f"  enriched: {jp.relative_to(PROCESSED_ROOT)}  jurisdiction_id={jid}  +{','.join(tags)}")
    print(
        f"\nMerge done: meeting_data={merged_meeting}, contacts={merged_contacts}, "
        f"no-match jurisdictions={missing_both}, unparseable files={skipped}."
    )

## 9) Cross-jurisdiction clustering *(optional)*

EmbeddingGemma finds similar policy language across Tuscaloosa and Big Timber. Needs `GEMINI_API_KEY` and prior demo JSON. Output: `04_embeddings/cross_jurisdiction_pairs.json`.


In [ ]:
# Demo 6 — EmbeddingGemma: cluster similar policy items across jurisdictions.
import json as _json

EMBED_MAX_ITEMS = int(os.environ.get("GOVERNANCE_EMBED_MAX_ITEMS", "200"))
EMBED_SIM_THRESHOLD = float(os.environ.get("GOVERNANCE_EMBED_SIM_THRESHOLD", "0.7"))
EMBED_REPORT_TOP_N = int(os.environ.get("GOVERNANCE_EMBED_REPORT_TOP_N", "30"))


def _harvest_snippets(json_path):
    """Pull high-signal text snippets from one Gemma JSON output."""
    try:
        data = _json.loads(json_path.read_text(encoding="utf-8"))
    except Exception:
        return []
    snippets = []
    payload = data
    if isinstance(data, dict) and isinstance(data.get("json_analysis"), dict):
        payload = data["json_analysis"]
    if not isinstance(payload, dict):
        return []
    # Demo 1 + 2 outputs carry raw_text / headlines.
    if isinstance(payload.get("raw_text"), str) and len(payload["raw_text"]) > 40:
        snippets.append(payload["raw_text"][:600])
    if isinstance(payload.get("headline"), str):
        snippets.append(payload["headline"])
    # Demo 3 outputs: subjects[], decisions[].
    for sub in payload.get("subjects", []) or []:
        name = sub.get("name") or sub.get("descriptive_name") or sub.get("subject_id")
        if name:
            snippets.append(str(name))
    for dec in payload.get("decisions", []) or []:
        title = dec.get("decision_title") or dec.get("title")
        if title:
            snippets.append(str(title))
    return [s.strip() for s in snippets if s and s.strip()]


items = []
for jp in sorted(GEMMA_JSON_ROOT.rglob("*.json")):
    if jp.name.startswith("_"):
        continue
    try:
        rel = jp.resolve().relative_to(GEMMA_JSON_ROOT.resolve())
    except ValueError:
        continue
    parts = rel.parts
    if len(parts) < 3:
        continue
    state, scope, slug = parts[0], parts[1], parts[2]
    jurisdiction = f"{state}/{scope}/{slug}"
    for text in _harvest_snippets(jp):
        items.append({"jurisdiction": jurisdiction, "source": str(rel), "text": text})
    if len(items) >= EMBED_MAX_ITEMS:
        break

items = items[:EMBED_MAX_ITEMS]
print(f"Embedding {len(items)} snippet(s) from {GEMMA_JSON_ROOT}")

if not items:
    print("Demo 6: no JSON snippets to embed yet — run Demos 1–3 first.")
else:
    if not GEMINI_API_KEY:
        raise RuntimeError("Demo 6 needs GEMINI_API_KEY (AI Studio) for EmbeddingGemma, or skip this cell.")
    vectors = embed_text_with_gemma(
        api_key=GEMINI_API_KEY,
        model=EMBEDDING_MODEL,
        texts=[it["text"] for it in items],
    )
    for it, v in zip(items, vectors):
        it["embedding_dims"] = len(v)

    embeddings_root = PIPE.root / "03_processed_outputs" / "04_embeddings"
    embeddings_root.mkdir(parents=True, exist_ok=True)
    items_path = embeddings_root / "items.jsonl"
    items_path.write_text(
        "\n".join(_json.dumps(it, ensure_ascii=False) for it in items) + "\n",
        encoding="utf-8",
    )

    sim = cosine_similarity_matrix(vectors)
    pairs = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i]["jurisdiction"] == items[j]["jurisdiction"]:
                continue  # we want CROSS-jurisdiction matches
            if sim[i][j] < EMBED_SIM_THRESHOLD:
                continue
            pairs.append({
                "similarity": round(float(sim[i][j]), 4),
                "left": {"jurisdiction": items[i]["jurisdiction"], "text": items[i]["text"][:200]},
                "right": {"jurisdiction": items[j]["jurisdiction"], "text": items[j]["text"][:200]},
            })
    pairs.sort(key=lambda p: -p["similarity"])
    pairs = pairs[:EMBED_REPORT_TOP_N]

    pairs_path = embeddings_root / "cross_jurisdiction_pairs.json"
    pairs_path.write_text(
        _json.dumps({"threshold": EMBED_SIM_THRESHOLD, "pairs": pairs}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"Demo 6 done — {len(pairs)} cross-jurisdiction pair(s) ≥ {EMBED_SIM_THRESHOLD}.")
    for p in pairs[:5]:
        print(f"  {p['similarity']:.3f}  {p['left']['jurisdiction']}  ↔  {p['right']['jurisdiction']}")
        print(f"            L: {p['left']['text'][:90]}")
        print(f"            R: {p['right']['text'][:90]}")
    print(f"  items   → {items_path.relative_to(PIPE.root)}")
    print(f"  pairs   → {pairs_path.relative_to(PIPE.root)}")


## 10) Safety review *(optional)*

ShieldGemma pass on Demo 4/5 outputs → `05_safety_review/`.


In [ ]:
# Safety Review — run ShieldGemma over Demo 4 audio summaries and Demo 5 image outputs.
import json as _json

SHIELD_MAX_FILES = int(os.environ.get("GOVERNANCE_SHIELD_MAX_FILES", "40"))

safety_root = PIPE.root / "03_processed_outputs" / "05_safety_review"
safety_root.mkdir(parents=True, exist_ok=True)

reviewed = []
flagged_count = 0
def _try_review(label, source_path, text_for_review):
    global flagged_count
    if not text_for_review or not text_for_review.strip():
        return
    try:
        if not GEMINI_API_KEY:
            print(f"  Skip shield ({label}): set GEMINI_API_KEY for AI Studio ShieldGemma.")
            return
        result = shield_review_text(
            api_key=GEMINI_API_KEY,
            model=SHIELD_MODEL,
            content=text_for_review[:4000],
            user_prompt=f"(automated {label} output from open-navigator pipeline)",
        )
    except Exception as e:
        print(f"  ! shield call failed for {source_path}: {e}")
        return
    rel = source_path.resolve().relative_to(GEMMA_JSON_ROOT.resolve()) if source_path.is_relative_to(GEMMA_JSON_ROOT) else Path(source_path.name)
    out = safety_root / rel.with_suffix(".shield.json")
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(_json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    reviewed.append({"source": str(rel), "flagged": result["flagged"], "categories": result["categories"]})
    if result["flagged"]:
        flagged_count += 1
        print(f"  ⚠ flagged: {rel} — {result['rationale']}")

# Demo 5 outputs — contact-image perception JSONs.
demo5_count = 0
for jp in sorted(GEMMA_JSON_ROOT.rglob("*.image_triage.json")):
    if demo5_count >= SHIELD_MAX_FILES // 2:
        break
    try:
        data = _json.loads(jp.read_text(encoding="utf-8"))
    except Exception:
        continue
    text = _json.dumps(data, ensure_ascii=False)
    _try_review("demo5_image_triage", jp, text)
    demo5_count += 1

# Demo 4 outputs — per-chunk policy analysis JSONs.
demo4_count = 0
for jp in sorted(GEMMA_JSON_ROOT.rglob("*.chunk_*.json")):
    if demo4_count >= SHIELD_MAX_FILES // 2:
        break
    try:
        data = _json.loads(jp.read_text(encoding="utf-8"))
    except Exception:
        continue
    text = _json.dumps(data, ensure_ascii=False)
    _try_review("demo4_audio_chunk", jp, text)
    demo4_count += 1

summary = {
    "model": SHIELD_MODEL,
    "reviewed_count": len(reviewed),
    "flagged_count": flagged_count,
    "reviewed": reviewed,
}
(safety_root / "_summary.json").write_text(
    _json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

if reviewed:
    print(f"\nShieldGemma review done — {len(reviewed)} item(s) reviewed, {flagged_count} flagged.")
    print(f"  summary → {(safety_root / '_summary.json').relative_to(PIPE.root)}")
else:
    print("Safety review: no Demo 4 / Demo 5 outputs found yet — run those demos first.")


## Troubleshooting (short)

| Issue | Fix |
|-------|-----|
| Looks stuck on Gatekeeper | E2B runs on **local GPU** (first file: load + infer). Watch `00_logs/gatekeeper_*.log`. Demos use **AI Studio** (26B). |
| Full model list spam | Keep `GOVERNANCE_LOG_LLM_CATALOG=0` |
| `count_triageable_files` missing | Runtime restart → re-run sections 1-3 |
| Gatekeeper OOM / CUDA | Use a Colab GPU runtime; E2B still needs VRAM |
| Demo 3 empty `.thoughts.md` | Try `GOVERNANCE_GENAI_MODEL=gemma-4-31b-it` on AI Studio |
| No data root | Sync pilots with `01_copy_scraped_meetings_cache_to_gdrive.py` |


## Where to look on Drive (after Run all)

```
03_processed_outputs/
  _gatekeeper/triage_report_*.json     ← what was kept vs excluded
  02_gemma_json/                       ← Demo 1–4 JSON + sidecars
  03_human_summaries/                  ← Smart Brevity (if generated)
```

**Live demo tip:** open `00_logs/gatekeeper_*.log` on Drive while §4 runs (large MT PDFs can take 1–2 min each).
